In [ ]:
import re, json, os
from collections import defaultdict, Counter
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# One-time download of the NLTK stop-word list
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')

os.makedirs('results', exist_ok=True)

## 2. Corpus
The original texts are kept untouched in `corpus`; every processed version is stored separately.

In [ ]:
corpus = {
    "D1":  "Information retrieval helps users find relevant documents.",
    "D2":  "Information Retrieval systems index documents and documents.",
    "D3":  "Search engines retrieve relevant web documents.",
    "D4":  "Document indexing supports efficient information retrieval.",
    "D5":  "Machine learning improves search engine ranking.",
    "D6":  "Search systems use indexed documents to retrieve information.",
    "D7":  "The search engine retrieves documents; the documents are searchable.",
    "D8":  "Indexing documents makes information retrieval more efficient.",
    "D9":  "DOCUMENTS contain information about document indexing.",
    "D10": "Retrieval systems organise documents and document collections.",
}

## Part 1 – Text preprocessing
### A. Inspecting the corpus

In [ ]:
for d, text in corpus.items():
    caps   = re.findall(r'\b[A-Z]\w*', text)                    # words containing an uppercase letter
    punct  = re.findall(r'[^\w\s]', text)                       # punctuation marks
    counts = Counter(re.sub(r'[^\w\s]', ' ', text.lower()).split())
    repeated = {w: c for w, c in counts.items() if c > 1}       # repeated terms
    print(f"{d:>3} | caps: {caps} | punct: {punct} | repeated: {repeated}")

 D1 | caps: ['Information'] | punct: ['.'] | repeated: {}
 D2 | caps: ['Information', 'Retrieval'] | punct: ['.'] | repeated: {'documents': 2}
 D3 | caps: ['Search'] | punct: ['.'] | repeated: {}
 D4 | caps: ['Document'] | punct: ['.'] | repeated: {}
 D5 | caps: ['Machine'] | punct: ['.'] | repeated: {}
 D6 | caps: ['Search'] | punct: ['.'] | repeated: {}
 D7 | caps: ['The'] | punct: [';', '.'] | repeated: {'the': 2, 'documents': 2}
 D8 | caps: ['Indexing'] | punct: ['.'] | repeated: {}
 D9 | caps: ['DOCUMENTS'] | punct: ['.'] | repeated: {}
D10 | caps: ['Retrieval'] | punct: ['.'] | repeated: {}


**Observations**
- **Uppercase/lowercase:** every document starts with a capital; `Retrieval` (D2), `DOCUMENTS` (D9) and `Retrieval` (D10) show that the same word appears in several case forms (`documents`/`DOCUMENTS`, `Information`/`information`).
- **Punctuation:** a full stop ends each sentence; D7 also contains a semicolon.
- **Repeated terms:** `documents` twice in D2, `the` and `documents` twice in D7.
- **Morphological variants:** document / documents; index / indexed / indexing; retrieve / retrieves / retrieval; search / searchable; engine / engines; system / systems.

### B. Basic steps implemented by hand (version A)

In [ ]:
def preprocess_document(text):
    """Lowercase, remove punctuation, tokenize on whitespace."""
    text = text.lower()                      # 1. lowercase
    text = re.sub(r'[^\w\s]', ' ', text)     # 2. replace punctuation by a space
    return text.split()                      # 3. whitespace tokenization

tokens_A = {d: preprocess_document(t) for d, t in corpus.items()}
for d, toks in tokens_A.items():
    print(d, toks)

D1 ['information', 'retrieval', 'helps', 'users', 'find', 'relevant', 'documents']
D2 ['information', 'retrieval', 'systems', 'index', 'documents', 'and', 'documents']
D3 ['search', 'engines', 'retrieve', 'relevant', 'web', 'documents']
D4 ['document', 'indexing', 'supports', 'efficient', 'information', 'retrieval']
D5 ['machine', 'learning', 'improves', 'search', 'engine', 'ranking']
D6 ['search', 'systems', 'use', 'indexed', 'documents', 'to', 'retrieve', 'information']
D7 ['the', 'search', 'engine', 'retrieves', 'documents', 'the', 'documents', 'are', 'searchable']
D8 ['indexing', 'documents', 'makes', 'information', 'retrieval', 'more', 'efficient']
D9 ['documents', 'contain', 'information', 'about', 'document', 'indexing']
D10 ['retrieval', 'systems', 'organise', 'documents', 'and', 'document', 'collections']


### C. Stop-word removal (version B)

In [ ]:
stop_words = set(stopwords.words('english'))   # NLTK list; the filtering itself is ours

def remove_stopwords(tokens):
    return [w for w in tokens if w not in stop_words]

tokens_B = {d: remove_stopwords(toks) for d, toks in tokens_A.items()}
for d, toks in tokens_B.items():
    print(d, toks)

D1 ['information', 'retrieval', 'helps', 'users', 'find', 'relevant', 'documents']
D2 ['information', 'retrieval', 'systems', 'index', 'documents', 'documents']
D3 ['search', 'engines', 'retrieve', 'relevant', 'web', 'documents']
D4 ['document', 'indexing', 'supports', 'efficient', 'information', 'retrieval']
D5 ['machine', 'learning', 'improves', 'search', 'engine', 'ranking']
D6 ['search', 'systems', 'use', 'indexed', 'documents', 'retrieve', 'information']
D7 ['search', 'engine', 'retrieves', 'documents', 'documents', 'searchable']
D8 ['indexing', 'documents', 'makes', 'information', 'retrieval', 'efficient']
D9 ['documents', 'contain', 'information', 'document', 'indexing']
D10 ['retrieval', 'systems', 'organise', 'documents', 'document', 'collections']


### D. Porter stemming (version C)
Porter stemming is a rule-based suffix stripper, **not** lemmatization: its output is not always a real word.

In [ ]:
stemmer = PorterStemmer()

def stem_tokens(tokens):
    return [stemmer.stem(w) for w in tokens]

tokens_C = {d: stem_tokens(toks) for d, toks in tokens_B.items()}
for d, toks in tokens_C.items():
    print(d, toks)

# Which original words were merged by the stemmer?
groups = defaultdict(set)
for toks in tokens_B.values():
    for w in toks:
        groups[stemmer.stem(w)].add(w)
print("\nStems that merge several word forms:")
for s, ws in sorted(groups.items()):
    if len(ws) > 1:
        print(f"  {s:10s} <- {sorted(ws)}")

D1 ['inform', 'retriev', 'help', 'user', 'find', 'relev', 'document']
D2 ['inform', 'retriev', 'system', 'index', 'document', 'document']
D3 ['search', 'engin', 'retriev', 'relev', 'web', 'document']
D4 ['document', 'index', 'support', 'effici', 'inform', 'retriev']
D5 ['machin', 'learn', 'improv', 'search', 'engin', 'rank']
D6 ['search', 'system', 'use', 'index', 'document', 'retriev', 'inform']
D7 ['search', 'engin', 'retriev', 'document', 'document', 'searchabl']
D8 ['index', 'document', 'make', 'inform', 'retriev', 'effici']
D9 ['document', 'contain', 'inform', 'document', 'index']
D10 ['retriev', 'system', 'organis', 'document', 'document', 'collect']

Stems that merge several word forms:
  document   <- ['document', 'documents']
  engin      <- ['engine', 'engines']
  index      <- ['index', 'indexed', 'indexing']
  retriev    <- ['retrieval', 'retrieve', 'retrieves']


In [ ]:
versions = {'A': tokens_A, 'B': tokens_B, 'C': tokens_C}   # the three versions of the collection

## Part 2 – Vocabulary and document representation

In [ ]:
def build_vocabulary(tokenized_documents):
    """Unique terms in alphabetical order (reproducible)."""
    vocab = set()
    for toks in tokenized_documents.values():
        vocab.update(toks)
    return sorted(vocab)

def build_term_document_matrix(tokenized_documents):
    """Binary matrix. rows: terms (sorted); columns: document IDs; cell = 1 if term occurs, else 0."""
    terms = build_vocabulary(tokenized_documents)
    doc_ids = list(tokenized_documents.keys())
    matrix = []
    for term in terms:
        matrix.append([1 if term in tokenized_documents[d] else 0 for d in doc_ids])
    return terms, doc_ids, matrix

def build_tf_matrix(tokenized_documents):
    """Term-frequency matrix: cell = number of occurrences of the term in the document."""
    terms = build_vocabulary(tokenized_documents)
    doc_ids = list(tokenized_documents.keys())
    matrix = []
    for term in terms:
        row = []
        for d in doc_ids:
            count = 0
            for tok in tokenized_documents[d]:      # we count ourselves (no Counter / pandas)
                if tok == term:
                    count += 1
            row.append(count)
        matrix.append(row)
    return terms, doc_ids, matrix

def to_dataframe(terms, doc_ids, matrix):
    """pandas is used ONLY for display / CSV export."""
    return pd.DataFrame(matrix, index=terms, columns=doc_ids)

vocab, bin_mat, tf_mat = {}, {}, {}
for v, toks in versions.items():
    vocab[v] = build_vocabulary(toks)
    bin_mat[v] = to_dataframe(*build_term_document_matrix(toks))
    tf_mat[v]  = to_dataframe(*build_tf_matrix(toks))
    print(f"Version {v}: vocabulary size = {len(vocab[v])}")
print(vocab['A'])

Version A: vocabulary size = 36
Version B: vocabulary size = 30
Version C: vocabulary size = 24
['about', 'and', 'are', 'collections', 'contain', 'document', 'documents', 'efficient', 'engine', 'engines', 'find', 'helps', 'improves', 'index', 'indexed', 'indexing', 'information', 'learning', 'machine', 'makes', 'more', 'organise', 'ranking', 'relevant', 'retrieval', 'retrieve', 'retrieves', 'search', 'searchable', 'supports', 'systems', 'the', 'to', 'use', 'users', 'web']


### Binary matrix and TF matrix – version A

In [ ]:
bin_mat['A']

,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
about,0,0,0,0,0,0,0,0,1,0
and,0,1,0,0,0,0,0,0,0,1
are,0,0,0,0,0,0,1,0,0,0
collections,0,0,0,0,0,0,0,0,0,1
contain,0,0,0,0,0,0,0,0,1,0
document,0,0,0,1,0,0,0,0,1,1
documents,1,1,1,0,0,1,1,1,1,1
efficient,0,0,0,1,0,0,0,1,0,0
engine,0,0,0,0,1,0,1,0,0,0
engines,0,0,1,0,0,0,0,0,0,0


In [ ]:
tf_mat['A']

,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
about,0,0,0,0,0,0,0,0,1,0
and,0,1,0,0,0,0,0,0,0,1
are,0,0,0,0,0,0,1,0,0,0
collections,0,0,0,0,0,0,0,0,0,1
contain,0,0,0,0,0,0,0,0,1,0
document,0,0,0,1,0,0,0,0,1,1
documents,1,2,1,0,0,1,2,1,1,1
efficient,0,0,0,1,0,0,0,1,0,0
engine,0,0,0,0,1,0,1,0,0,0
engines,0,0,1,0,0,0,0,0,0,0


### Version B

In [ ]:
bin_mat['B']

,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
collections,0,0,0,0,0,0,0,0,0,1
contain,0,0,0,0,0,0,0,0,1,0
document,0,0,0,1,0,0,0,0,1,1
documents,1,1,1,0,0,1,1,1,1,1
efficient,0,0,0,1,0,0,0,1,0,0
engine,0,0,0,0,1,0,1,0,0,0
engines,0,0,1,0,0,0,0,0,0,0
find,1,0,0,0,0,0,0,0,0,0
helps,1,0,0,0,0,0,0,0,0,0
improves,0,0,0,0,1,0,0,0,0,0


In [ ]:
tf_mat['B']

,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
collections,0,0,0,0,0,0,0,0,0,1
contain,0,0,0,0,0,0,0,0,1,0
document,0,0,0,1,0,0,0,0,1,1
documents,1,2,1,0,0,1,2,1,1,1
efficient,0,0,0,1,0,0,0,1,0,0
engine,0,0,0,0,1,0,1,0,0,0
engines,0,0,1,0,0,0,0,0,0,0
find,1,0,0,0,0,0,0,0,0,0
helps,1,0,0,0,0,0,0,0,0,0
improves,0,0,0,0,1,0,0,0,0,0


### Version C

In [ ]:
bin_mat['C']

,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
collect,0,0,0,0,0,0,0,0,0,1
contain,0,0,0,0,0,0,0,0,1,0
document,1,1,1,1,0,1,1,1,1,1
effici,0,0,0,1,0,0,0,1,0,0
engin,0,0,1,0,1,0,1,0,0,0
find,1,0,0,0,0,0,0,0,0,0
help,1,0,0,0,0,0,0,0,0,0
improv,0,0,0,0,1,0,0,0,0,0
index,0,1,0,1,0,1,0,1,1,0
inform,1,1,0,1,0,1,0,1,1,0


In [ ]:
tf_mat['C']

,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
collect,0,0,0,0,0,0,0,0,0,1
contain,0,0,0,0,0,0,0,0,1,0
document,1,2,1,1,0,1,2,1,2,2
effici,0,0,0,1,0,0,0,1,0,0
engin,0,0,1,0,1,0,1,0,0,0
find,1,0,0,0,0,0,0,0,0,0
help,1,0,0,0,0,0,0,0,0,0
improv,0,0,0,0,1,0,0,0,0,0
index,0,1,0,1,0,1,0,1,1,0
inform,1,1,0,1,0,1,0,1,1,0


**Binary vs term-frequency.**
- The **binary matrix** only records *presence/absence* of a term in a document (0/1): it answers "does the document contain the term?" (Boolean retrieval).
- The **TF matrix** records *how many times* the term occurs, so it keeps the importance of repeated words (e.g. `documents` = 2 in D2 and D7) – the basis of ranked retrieval (TF-IDF, BM25). The binary matrix can always be derived from the TF matrix (`TF > 0`), not the reverse.

## Part 3 – Inverted index with positions
### 1. Manual index for D1–D3 (version A)
Built by hand from the token lists (zero-based positions):

- D1: information(0) retrieval(1) helps(2) users(3) find(4) relevant(5) documents(6)
- D2: information(0) retrieval(1) systems(2) index(3) documents(4) and(5) documents(6)
- D3: search(0) engines(1) retrieve(2) relevant(3) web(4) documents(5)

In [ ]:
manual_index = {
    'information': {'D1': [0], 'D2': [0]},
    'retrieval':   {'D1': [1], 'D2': [1]},
    'documents':   {'D1': [6], 'D2': [4, 6], 'D3': [5]},
}

### 2–4. Automatic index and `search_term()`

In [ ]:
def build_inverted_index(tokenized_documents):
    """term -> document ID -> list of zero-based positions."""
    index = defaultdict(lambda: defaultdict(list))
    for doc_id, tokens in tokenized_documents.items():
        for position, term in enumerate(tokens):      # enumerate gives the 0-based position
            index[term][doc_id].append(position)
    # convert to plain dicts with sorted terms (clean JSON / reproducible)
    return {t: dict(index[t]) for t in sorted(index)}

def search_term(inverted_index, term):
    """Return the list of document IDs containing `term` (empty list if unknown)."""
    if term not in inverted_index:
        return []
    return list(inverted_index[term].keys())

inv_index = {v: build_inverted_index(toks) for v, toks in versions.items()}
print(json.dumps(inv_index['A'], indent=1)[:900], "...")

{
 "about": {
  "D9": [
   3
  ]
 },
 "and": {
  "D2": [
   5
  ],
  "D10": [
   4
  ]
 },
 "are": {
  "D7": [
   7
  ]
 },
 "collections": {
  "D10": [
   6
  ]
 },
 "contain": {
  "D9": [
   1
  ]
 },
 "document": {
  "D4": [
   0
  ],
  "D9": [
   4
  ],
  "D10": [
   5
  ]
 },
 "documents": {
  "D1": [
   6
  ],
  "D2": [
   4,
   6
  ],
  "D3": [
   5
  ],
  "D6": [
   4
  ],
  "D7": [
   4,
   6
  ],
  "D8": [
   1
  ],
  "D9": [
   0
  ],
  "D10": [
   3
  ]
 },
 "efficient": {
  "D4": [
   3
  ],
  "D8": [
   6
  ]
 },
 "engine": {
  "D5": [
   4
  ],
  "D7": [
   2
  ]
 },
 "engines": {
  "D3": [
   1
  ]
 },
 "find": {
  "D1": [
   4
  ]
 },
 "helps": {
  "D1": [
   2
  ]
 },
 "improves": {
  "D5": [
   2
  ]
 },
 "index": {
  "D2": [
   3
  ]
 },
 "indexed": {
  "D6": [
   3
  ]
 },
 "indexing": {
  "D4": [
   1
  ],
  "D8": [
   0
  ],
  "D9": [
   5
  ]
 },
 "information": { ...


### 5. Manual vs automatic index (D1–D3)

In [ ]:
# Restrict the automatic index (version A) to D1-D3 and to the 3 terms of the manual index
tokens_A_d1_d3 = {d: tokens_A[d] for d in ['D1', 'D2', 'D3']}
auto_small = build_inverted_index(tokens_A_d1_d3)
for term in manual_index:
    print(f"{term:12s} manual: {manual_index[term]}")
    print(f"{'':12s} auto  : {auto_small[term]}   -> {'OK' if manual_index[term] == auto_small[term] else 'DIFFERENT'}")
assert all(manual_index[t] == auto_small[t] for t in manual_index)

information  manual: {'D1': [0], 'D2': [0]}
             auto  : {'D1': [0], 'D2': [0]}   -> OK
retrieval    manual: {'D1': [1], 'D2': [1]}
             auto  : {'D1': [1], 'D2': [1]}   -> OK
documents    manual: {'D1': [6], 'D2': [4, 6], 'D3': [5]}
             auto  : {'D1': [6], 'D2': [4, 6], 'D3': [5]}   -> OK


No discrepancy. Note that in version A, `retrieve` (D3) is a *different* term from `retrieval`, so D3 is not in the `retrieval` posting list. With stemming (version C) both become `retriev` – see Part 5.

## Part 4 – Effect of preprocessing

In [ ]:
def index_stats(tokenized_documents, index):
    n_docs = len(tokenized_documents)
    total_tokens = sum(len(t) for t in tokenized_documents.values())
    return {
        'Vocabulary size': len(index),
        'Avg tokens / document': round(total_tokens / n_docs, 2),
        'Total postings (term-doc pairs)': sum(len(docs) for docs in index.values()),
        'Total stored positions': sum(len(p) for docs in index.values() for p in docs.values()),
    }

labels = {'A': 'A: lowercase + punctuation', 'B': 'B: A + stop-words', 'C': 'C: B + stemming'}
comparison = pd.DataFrame({labels[v]: index_stats(versions[v], inv_index[v]) for v in versions}).T
comparison.index.name = 'Version'
for col in comparison.columns:
    if col != 'Avg tokens / document':
        comparison[col] = comparison[col].astype(int)
comparison

,Vocabulary size,Avg tokens / document,Total postings (term-doc pairs),Total stored positions
Version,,,,
A: lowercase + punctuation,36,6.9,66,69
B: A + stop-words,30,6.1,59,61
C: B + stemming,24,6.1,57,61


### Interpretation (117 words)
Preprocessing shrinks the index step by step. Lowercasing and punctuation removal (A) already merge forms such as `Documents`/`DOCUMENTS`, giving 36 terms. Stop-word removal (B) deletes frequent, low-information words (`and`, `the`, `to`, `about`): the vocabulary drops to 30, tokens per document from 6.9 to 6.1, postings from 66 to 59 and stored positions from 69 to 61. Stemming (C) does not remove tokens (still 61 positions) but merges variants (`retrieval`/`retrieve`, `index`/`indexing`, `document`/`documents`), reducing the vocabulary to 24 and postings to 57. A smaller index is cheaper and increases recall, but not every step is harmless: stop-word removal shifts positions and breaks exact phrase queries, and stemming can conflate different meanings and produce non-words (`effici`), so precision may fall.

## Part 5 – Verification

### 1–2. `search_term()` vs brute force, and the four required cases

In [ ]:
def brute_force(tokenized_documents, term):
    return [d for d, toks in tokenized_documents.items() if term in toks]

# 1. search_term must return exactly the documents whose token lists contain the term (every term, every version)
for v in versions:
    for term in vocab[v]:
        assert search_term(inv_index[v], term) == brute_force(versions[v], term), (v, term)
    assert search_term(inv_index[v], 'blockchain') == []
print("Check 1 OK: search_term == brute force for every term of A, B and C")

# 2. Four required cases (version B, plus a stemming comparison B vs C)
print("\nCase 1 – one document only      : 'machine'    ->", search_term(inv_index['B'], 'machine'))
print("Case 2 – several documents       : 'documents'  ->", search_term(inv_index['B'], 'documents'))
print("Case 3 – unknown term            : 'blockchain' ->", search_term(inv_index['B'], 'blockchain'))

print("\nCase 4 – form changed by stemming:")
print("  B  'retrieval' ->", search_term(inv_index['B'], 'retrieval'))
print("  C  'retrieval' ->", search_term(inv_index['C'], 'retrieval'), "(raw form no longer in the index!)")
q = stemmer.stem('retrieval')
print(f"  C  '{q}'  ->", search_term(inv_index['C'], q), "(stemmed query: also finds retrieve/retrieves)")
print("  B  'documents' ->", search_term(inv_index['B'], 'documents'))
print("  C  'document'  ->", search_term(inv_index['C'], stemmer.stem('documents')))

assert search_term(inv_index['B'], 'machine') == ['D5']
assert len(search_term(inv_index['B'], 'documents')) > 1
assert search_term(inv_index['B'], 'blockchain') == []
assert search_term(inv_index['C'], 'retrieval') == []
assert set(search_term(inv_index['B'], 'retrieval')) < set(search_term(inv_index['C'], q))
print("\nChecks 2 OK")

Check 1 OK: search_term == brute force for every term of A, B and C

Case 1 – one document only      : 'machine'    -> ['D5']
Case 2 – several documents       : 'documents'  -> ['D1', 'D2', 'D3', 'D6', 'D7', 'D8', 'D9', 'D10']
Case 3 – unknown term            : 'blockchain' -> []

Case 4 – form changed by stemming:
  B  'retrieval' -> ['D1', 'D2', 'D4', 'D8', 'D10']
  C  'retrieval' -> [] (raw form no longer in the index!)
  C  'retriev'  -> ['D1', 'D2', 'D3', 'D4', 'D6', 'D7', 'D8', 'D10'] (stemmed query: also finds retrieve/retrieves)
  B  'documents' -> ['D1', 'D2', 'D3', 'D6', 'D7', 'D8', 'D9', 'D10']
  C  'document'  -> ['D1', 'D2', 'D3', 'D4', 'D6', 'D7', 'D8', 'D9', 'D10']

Checks 2 OK


Case 4 shows the key practical point: **the query must go through the same preprocessing as the documents**. Stemming increases recall (D3, D6, D7 contain `retrieve(s)` and are now found) but the query `retrieval` unstemmed finds nothing.

### 3. Positions correspond to zero-based positions in the processed token lists

In [ ]:
for v in versions:
    n_positions = 0
    for term, postings in inv_index[v].items():
        for doc_id, positions in postings.items():
            assert positions == sorted(positions)
            for p in positions:
                assert versions[v][doc_id][p] == term, (v, term, doc_id, p)   # token at position p IS the term
                n_positions += 1
    # every token of the collection is stored exactly once
    assert n_positions == sum(len(t) for t in versions[v].values())
print("Check 3 OK: every stored position points to the right token; #positions == #tokens in A, B, C")

# Example: positions move after stop-word removal (D2: 'documents' was at [4, 6] in A)
print("D2 'documents' in A:", inv_index['A']['documents']['D2'], "| in B:", inv_index['B']['documents']['D2'])
print("D2 tokens A:", tokens_A['D2']); print("D2 tokens B:", tokens_B['D2'])

Check 3 OK: every stored position points to the right token; #positions == #tokens in A, B, C
D2 'documents' in A: [4, 6] | in B: [4, 5]
D2 tokens A: ['information', 'retrieval', 'systems', 'index', 'documents', 'and', 'documents']
D2 tokens B: ['information', 'retrieval', 'systems', 'index', 'documents', 'documents']


Positions in B differ from A because stop-words (`and`) are removed **before** indexing, so the following tokens shift left. In C the positions equal those of B (stemming replaces tokens, it does not remove any) but the stored terms are stems.

### 4. Manual index D1–D3 vs automatic index (already compared in Part 3)

In [ ]:
# Same comparison again, with the index built from the FULL collection restricted to D1-D3
for term in manual_index:
    restricted = {d: p for d, p in inv_index['A'][term].items() if d in ('D1', 'D2', 'D3')}
    assert restricted == manual_index[term], term
print("Check 4 OK: manual index == automatic index for 'information', 'retrieval', 'documents' (D1-D3)")

Check 4 OK: manual index == automatic index for 'information', 'retrieval', 'documents' (D1-D3)


### 5. Matrix values

In [ ]:
for v in versions:
    B, T = bin_mat[v], tf_mat[v]
    assert set(B.values.flatten()) <= {0, 1}                           # binary: only 0 and 1
    assert ((T > 0).astype(int).values == B.values).all()               # binary == (TF > 0)
    for d in T.columns:                                                  # column sums = number of tokens
        assert T[d].sum() == len(versions[v][d])
    for d, toks in versions[v].items():                                  # independent check with Counter
        c = Counter(toks)
        for term in T.index:
            assert T.loc[term, d] == c.get(term, 0)
print("Check 5 OK: binary only 0/1, TF = number of occurrences, binary == (TF>0), column sums == document lengths")
print("Example TF(documents, D2) in A =", tf_mat['A'].loc['documents', 'D2'], "| binary =", bin_mat['A'].loc['documents', 'D2'])

Check 5 OK: binary only 0/1, TF = number of occurrences, binary == (TF>0), column sums == document lengths
Example TF(documents, D2) in A = 2 | binary = 1


## Export of results (Part 6)

In [ ]:
for v in versions:
    with open(f'results/tokens_{v}.json', 'w', encoding='utf-8') as f:
        json.dump(versions[v], f, indent=2, ensure_ascii=False)
    with open(f'results/vocabulary_{v}.txt', 'w', encoding='utf-8') as f:
        f.write("\n".join(vocab[v]) + "\n")
    bin_mat[v].to_csv(f'results/binary_matrix_{v}.csv', index_label='term')
    tf_mat[v].to_csv(f'results/tf_matrix_{v}.csv', index_label='term')
    with open(f'results/inverted_index_{v}.json', 'w', encoding='utf-8') as f:
        json.dump(inv_index[v], f, indent=2, ensure_ascii=False)
comparison.to_csv('results/preprocessing_comparison.csv')
print(sorted(os.listdir('results')))

['binary_matrix_A.csv', 'binary_matrix_B.csv', 'binary_matrix_C.csv', 'inverted_index_A.json', 'inverted_index_B.json', 'inverted_index_C.json', 'preprocessing_comparison.csv', 'tf_matrix_A.csv', 'tf_matrix_B.csv', 'tf_matrix_C.csv', 'tokens_A.json', 'tokens_B.json', 'tokens_C.json', 'vocabulary_A.txt', 'vocabulary_B.txt', 'vocabulary_C.txt']
